# Census Tract Data
## Imports

In [ ]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from shapely.geometry import Point

DER_THRESHOLD = 5

# Storage Data

In [ ]:
# storage = pd.read_excel("Map_Zip_Pie_data.xlsx")
storage = pd.read_excel("Storage_LatLong.xlsx")

# Make names more descriptive
rename_map = {
    "Utility": "utility_name",
    "Nameplate Capacity (MW)": "nameplate_capacity_mw",
    "Nameplate Capacity (in KW AC)": "nameplate_capacity_kw_ac",
    "Fuel Type": "fuel_type",
    "Facility City": "city",
    "County": "county",
    "CAISO Flag": "caiso_flag",
    "Facility Zip": "zip_code",
    "Customer Sector": "sector",
    "Approval Date": "approval_date",
    'Technology Type': 'technology_type',
    'Disconnect Date': "disconnect_date",
    'OG Reported Technology Type': "og_reported_technology_type",
    'Count of Nameplate Capacity (in KW AC)': "count_of_nameplate_capacity_kw_ac",
    'Latitude (generated)': 'latitude',
    'Longitude (generated)': 'longitude'
}
storage.rename(columns=rename_map, inplace=True)
storage["nameplate_capacity_mw"] = storage["nameplate_capacity_kw_ac"] / 1000

# Filter only DER as being those in residential or commercial sector with less than 10 mW or 10000 kW capacity
storage_der = storage[storage["sector"].isin(["Residential", "Commercial"])]
storage_der = storage_der[storage_der["nameplate_capacity_mw"] <= DER_THRESHOLD]

storage_utility = storage[storage["nameplate_capacity_mw"] > DER_THRESHOLD]
storage_der["zip_code"] = storage_der["zip_code"].astype(str).str.zfill(5).str.strip()

print(storage.columns)

In [ ]:
tract_shapefile = 'tl_2024_06_tract/tl_2024_06_tract.shp'
gdf_tracts = gpd.read_file(tract_shapefile)
gdf_tracts = gdf_tracts.to_crs(epsg=4326)
geometry = [Point(xy) for xy in zip(storage_der['longitude'], storage_der['latitude'])]
gdf_points = gpd.GeoDataFrame(storage_der, geometry=geometry, crs='EPSG:4326')
gdf_with_tracts = gpd.sjoin(gdf_points, gdf_tracts, how='left', predicate='within')
storage_der = gdf_with_tracts.rename(columns={'GEOID': 'census_tract'})
print(storage_der.head())
# Merge the points and tracts dataframes
# merged_df = pd.merge(storage_der, gdf_with_tracts[['census_tract']], how='left', on='zip_code')

In [ ]:
print(storage_der.info())
nan_zip_rows = storage_der[storage_der['census_tract'].isna()]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")

nan_zip_pct = len(nan_zip_rows) / len(storage_der) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")
if 'nameplate_capacity_mw' in storage_der.columns:
    capacity_total = storage_der['nameplate_capacity_mw'].sum()
    capacity_lost = nan_zip_rows['nameplate_capacity_mw'].sum()
    print(f"\nCapacity lost if removed: {capacity_lost:.2f} kW "
          f"({capacity_lost / capacity_total * 100:.2f}% of total)")
else:
    print("\nColumn 'capacity_kw' not found, check column names for capacity.")

# Technology type distribution overall
print("\nTechnology mix (overall):")
print(storage_der['fuel_type'].value_counts(normalize=True))

# Technology mix for NaN ZIP rows
print("\nTechnology mix (NaN ZIP only):")
print(nan_zip_rows['fuel_type'].value_counts(normalize=True))

# Check how many unique ZIP codes are present
valid_zip_codes = storage_der.loc[~storage_der['census_tract'].isna(), 'census_tract'].nunique()
print(f"\nNumber of unique valid ZIP codes: {valid_zip_codes}")

# Check top 10 ZIP codes by capacity
if 'nameplate_capacity_mw' in storage_der.columns:
    print("\nTop 10 ZIP codes by total capacity:")
    print(storage_der.groupby('census_tract')['nameplate_capacity_mw'].sum().sort_values(ascending=False).head(10))

print(sorted(storage_der["census_tract"].unique()))

print(storage_der.head())

# Wind Data

In [ ]:
# Wind data from: https://energy.usgs.gov/uswtdb/
# Smallest granularity is lat/long -> can be converted to census tract with shp file
uswtdb_wind = pd.read_csv("USWTDB_wind_data.csv")
uswtdb_wind.rename(columns={'t_county': 'County'}, inplace=True)
uswtdb_wind['County'] = uswtdb_wind['County'].str.strip().str.lower()
uswtdb_wind["County"] = uswtdb_wind["County"].str.replace(" county", "", regex=False)
uswtdb_wind['FIPS'] = uswtdb_wind['County'].map(name_to_fips)
uswtdb_wind['FIPS'] = uswtdb_wind['FIPS'].astype(str).str.zfill(5)
uswtdb_wind = uswtdb_wind.drop(columns=["t_fips"])
rename_map = {
    # IDs & references
    "case_id":          "case_id",                       # unique row ID (keep as‑is)
    "faa_ors":          "faa_ors_id",                    # FAA Obstruction Repository System ID
    "faa_asn":          "faa_asn_id",                    # FAA Aeronautical Study Number
    "usgs_pr_id":       "usgs_project_id",
    "eia_id":           "eia_id",

    # Location
    "t_state":          "state",
    "County":           "county",
    "FIPS":             "fips",                     # duplicate field in raw file
    "xlong":            "longitude",
    "ylat":             "latitude",

    # Project‑level info
    "p_name":           "project_name",
    "p_year":           "project_year",
    "p_tnum":           "project_turbine_count",
    "p_cap":            "project_capacity_mw",

    # Turbine‑level info
    "t_manu":           "turbine_manufacturer",
    "t_model":          "turbine_model",
    "t_cap":            "turbine_capacity_kw",
    "t_hh":             "turbine_hub_height_m",
    "t_rd":             "turbine_rotor_diameter_m",
    "t_rsa":            "turbine_rotor_swept_area_m2",
    "t_ttlh":           "turbine_total_tip_height_m",
    "t_retrofit":       "turbine_retrofit_flag",
    "t_retro_yr":       "turbine_retrofit_year",
    "t_offshore":       "offshore_flag",

    # Data quality & imagery
    "t_conf_atr":       "attribute_confidence",
    "t_conf_loc":       "location_confidence",
    "t_img_date":       "image_date",
    "t_img_src":        "image_source",
}

# apply the rename
uswtdb_wind = uswtdb_wind.rename(columns=rename_map)

# loads shp file to map lat/lon to zip code
gdf_wind = gpd.GeoDataFrame(
    uswtdb_wind,
    geometry=gpd.points_from_xy(uswtdb_wind["longitude"], uswtdb_wind["latitude"]),
    crs="EPSG:4326"
)
joined = gpd.sjoin(gdf_wind, gdf_tracts, how="left", predicate="within")
uswtdb_wind["census_tract"] = joined["GEOID"].astype(str).str.zfill(5)
uswtdb_wind["census_tract"] = uswtdb_wind["census_tract"].astype(str).str.strip()

uswtdb_wind = uswtdb_wind[uswtdb_wind['state'].str.startswith('CA', na=False)].copy()
uswtdb_wind_der = uswtdb_wind[uswtdb_wind['project_capacity_mw'] <= DER_THRESHOLD].copy()
uswtdb_wind_der.to_csv('USWTDB_CA_Under_5MW.csv', index=False)

uswtdb_wind_utility = uswtdb_wind[uswtdb_wind['project_capacity_mw'] > DER_THRESHOLD].copy()